[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/geometry/sundial/sundial.ipynb)

# Sundials

This example computes the hour lines and seasonal shadow paths of a horizontal sundial. A sundial tells the time by the shadow of a gnomon whose inclined edge, called the style, points at the celestial pole around which the sky turns once a day. As the sun turns with the sky, the style's shadow sweeps from one fixed hour line to the next, while the shadow of the gnomon's tip, called the nodus, traces a daily curve that shifts with the season.

Each direction to the sun is modelled as an intrinsic antivector produced by two rotor turns `(hour_turns * declination_turns) >> equator` of the celestial equator's highest point. Pairing any point with the horizontal dial plane `dial & point` reads its height above the dial, placing the nodus shadow at `nodus - suns * ((dial & nodus) / (dial & suns))`. Over a single day the sun's rays keep a constant angle to the pole and sweep a quadric cone modelled as the duality map `across * (across & Point) - (across & sun).squared() * Point.dual_inverse()`, whose cut with the dial plane forms the season's conic shadow path.

In spherical-astronomy and gnomonics notation, the solar altitude $h$ and azimuth $A$ at latitude $\varphi$, declination $\delta$, and hour angle $H$ satisfy $\sin h = \sin\varphi\sin\delta + \cos\varphi\cos\delta\cos H$ and $\tan A = \cos\delta\sin H / (\cos\varphi\sin\delta - \sin\varphi\cos\delta\cos H)$, each horizontal hour line makes the angle $\theta = \operatorname{atan2}(\sin\varphi\sin H, \cos H)$ with the noon line, and eliminating $H$ across the projection $(x, y) = h_{\mathrm{nodus}}\cot h\,(\sin A, \cos A)$ yields separate rational trigonometric equations for the hyperbolic and rectilinear declination curves.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np

from numga import NumpyContext
from numga.algebras import VGA3D as ga
from examples.util.animation import save_figure, show_animation
from examples.geometry.sundial import render

mv = NumpyContext(ga).multivector
# Directions to the sun, the pole, and the compass points, together with points on the dial, are antivectors;
# planes through the origin are vectors.
Point = ga.gatype.antivector()                      # Antivector-
Plane = ga.gatype.vector()                          # Vector+
Rotor = ga.gatype.rotor()                           # Rotor+
# A quadric cone maps each ray to its polar plane through the origin.
Cone = ga.gatype((Plane, Point))                    # Plane+ <- Point-

# Local compass frame at the dial and horizontal dial plane through the origin.
east, north, up = mv.basis().dual()                 # [] Point-, Point-, Point-
west = -east                                        # [] Point-
dial = mv.z                                         # [] Plane+
# Solar noon and the sun's hour angle per hour, a twenty-fourth of a turn.
noon = 12
turn_per_hour = 2 * np.pi / 24

## Two turns of the sun

The celestial pole stands above north in the meridian plane by the latitude, and the celestial equator's highest point lies a quarter turn further south, where the sun crosses the meridian at noon on the equinoxes. Every other solar position turns that equinox point twice, first towards the pole in the meridian plane by the seasonal declination and then westwards about the pole by the hour angle. At solar noon the sun's altitude equals a quarter turn minus the latitude plus the declination.

In [ ]:
# Dial latitude and axial tilt (the sun's declination at the solstices).
latitude = np.radians(52.0)
tilt = np.radians(23.44)
solstices_and_equinox = np.array([-tilt, 0.0, tilt])

In [ ]:
def sky(latitude: float) -> tuple[Point, Point]:
    """The celestial pole lies at the latitude above north in the meridian plane, and the celestial equator's highest point lies a quarter turn further (`[] Point-`, `[] Point-`)."""
    # The commutator of two orthogonal unit directions is the bivector of the meridian plane they span.
    meridian = up.commutator(north)                                            # [] Bivector+
    pole_turn = (meridian * (-latitude / 2)).exp()                             # [] Rotor+
    equator_turn = (meridian * (-(latitude + np.pi / 2) / 2)).exp()            # [] Rotor+
    return pole_turn >> north, equator_turn >> north                           # [] Point-, Point-


def sun(pole: Point, equator: Point, declinations: np.ndarray, hour_angles: np.ndarray) -> Point:
    """Two turns of the celestial equator's highest point place the sun at `declinations` towards the pole and `hour_angles` westwards about the pole (`[...] Point-`)."""
    # Declination turns in the meridian plane, and hour angle turns in the plane of the celestial equator.
    declination_turns = (pole.commutator(equator) * (-declinations / 2)).exp() # [...] Rotor+
    hour_turns = (west.commutator(equator) * (-hour_angles / 2)).exp()         # [...] Rotor+
    turns = hour_turns * declination_turns                                     # [...] Rotor+
    return turns >> equator                                                    # [...] Point-


pole, equator = sky(latitude)                                                  # [] Point-, Point-
noon_suns = sun(pole, equator, solstices_and_equinox, np.zeros_like(solstices_and_equinox))  # [3] Point-
print("noon heights, winter, equinox, summer:", np.degrees(np.arcsin((dial & noon_suns).to_array())).round(2))

## Hour lines on the horizontal dial

The nodus casts its shadow where the ray from the sun past the nodus meets the dial plane. Because the style runs through the nodus towards the celestial pole, the plane spanned by the style and the sun at a fixed hour angle contains every seasonal declination turn at that hour. Each hour's shadow therefore falls along a single straight line radiating from the foot of the style, crowding near the noon line and spreading towards morning and evening.

In [ ]:
# Nodus height above the dial, drawn hours with their hour angles, and declinations sampled between the solstices.
height = 1.0
hours = np.arange(6, 19)
n_hours = len(hours)
hour_angles = (hours - noon) * turn_per_hour
n_declinations = 25

In [ ]:
def shadow(nodus: Point, suns: Point) -> Point:
    """Each ray from `suns` past `nodus` meets the horizontal dial plane where its height above the dial vanishes (`[...] Point-`)."""
    # Pairing a point with the horizontal dial plane reads its height above the dial.
    return nodus - suns * ((dial & nodus) / (dial & suns))                     # [...] Point-


nodus = up * height                                                            # [] Point-
# Foot where the style line along the celestial pole meets the horizontal dial.
foot = shadow(nodus, pole)                                                     # [] Point-
declinations = np.linspace(-tilt, tilt, n_declinations)                        # [n_declinations]
hour_suns = sun(pole, equator, declinations[None, :], hour_angles[:, None])    # [n_hours, n_declinations] Point-
hour_lines = shadow(nodus, hour_suns)                                          # [n_hours, n_declinations] Point-
offsets = (hour_lines - foot).normalized()                                      # [n_hours, n_declinations] Point-
east_offsets = mv.x & offsets                                                  # [n_hours, n_declinations] Scalar+
north_offsets = mv.y & offsets                                                 # [n_hours, n_declinations] Scalar+
print("hour line bearings from the noon line, in degrees:", np.degrees(np.arctan2(east_offsets[:, -1].to_array(), north_offsets[:, -1].to_array())).round(1))

## Seasonal shadow conics

Over a single day the ray from the nodus to the sun keeps a fixed angle to the celestial pole and sweeps a circular cone. The term ***duality map*** here means a linear map from intrinsic points to extrinsic planes, and a ray lies on the daily cone where its self-pairing `cones(ray) & ray` vanishes; the metric enters only through `Point.dual_inverse()`, which compares each ray's component along the pole against its squared length. Because that quadratic form depends only on the squared sine of the declination, its two nappes carry conjugate summer and winter declinations as the two branches of a single hyperbola, while at the equinoxes the cone flattens into the equatorial plane and meets the dial in a straight east-west line.

In [ ]:
# Seasonal declinations spaced one month apart between the solstices, and daily hour-angle samples.
n_months = 3
n_seasons = 2 * n_months + 1
day_hours = np.arange(3, 21.01, 0.05)
n_samples = len(day_hours)
day_angles = (day_hours - noon) * turn_per_hour

In [ ]:
def cone(pole: Point, sun: Point) -> Cone:
    """The daily cone of rays about `pole` through `sun` maps each ray to its polar plane, holding both nappes at opposite declinations (`[...] Plane+ <- Point-`)."""
    # The plane perpendicular to the pole reads a unit ray's sine of declination along the pole.
    across = pole.dual_inverse()                                               # [...] Plane+
    # A ray lies on the cone where its height squared along the pole matches the sun's times its length squared.
    return across * (across & Point) - (across & sun).squared() * Point.dual_inverse()  # [...] Plane+ <- Point-


seasons = tilt * np.sin(np.pi / 2 * np.arange(-n_months, n_months + 1) / n_months)  # [n_seasons]
days = sun(pole, equator, seasons[:, None], day_angles[None, :])               # [n_seasons, n_samples] Point-
rays = (nodus - shadow(nodus, days)).normalized()                              # [n_seasons, n_samples] Point-
cones = cone(pole, days[:, 0])                                                 # [n_seasons] Plane+ <- Point-
save_figure(render.dial_figure(nodus, foot, seasons, days, hour_suns, hours), "sundial");

Grey hour lines radiate from the dark gnomon base along the noon line to their numbered border marks, while the seasonal shadow paths share colours across conjugate summer and winter declinations so the yellow solstice hyperbola branches bracket the straight purple equinox line.

## Sweeping the daily ray cone through the year

As the solar declination swings from winter solstice to summer solstice, the cone of daily rays opens and closes about the celestial pole. In summer the steep cone cuts the horizontal plate in a tight southern hyperbola branch near the gnomon, at the equinoxes the flattened equatorial sheet traces the straight east-west line, and in winter the shallow cone sweeps the far northern hyperbola branch.

In [ ]:
# Half-year frame count (bounced forward and back), drawn ray hours, and frame duration.
n_frames = 108
n_half_year = n_frames // 2 + 1
ray_hours = np.arange(4, 21, 0.25)
n_ray_hours = len(ray_hours)
ray_angles = (ray_hours - noon) * turn_per_hour
frame_duration_ms = 40

In [ ]:
def year_states(
    pole: Point,
    equator: Point,
    declinations: np.ndarray,
    ray_angles: np.ndarray,
    day_angles: np.ndarray,
) -> Iterator[tuple[Point, Point]]:
    """Yields `(day_rays, day_suns)` (`[n_ray_hours] Point-`, `[n_samples] Point-`) for each declination in `declinations`."""
    for declination in declinations:
        yield sun(pole, equator, declination, ray_angles), sun(pole, equator, declination, day_angles)


year_declinations = tilt * np.sin(np.linspace(-np.pi / 2, np.pi / 2, n_half_year))  # [n_half_year]
envelope_rays = sun(pole, equator, year_declinations[:, None], ray_angles[None, :]) # [n_half_year, n_ray_hours] Point-
recorded_year: list[tuple[Point, Point]] = []


def streamed_year() -> Iterator[tuple[Point, Point]]:
    """Yields each half-year `(day_rays, day_suns)` state while recording it once for downstream checks."""
    for state in year_states(pole, equator, year_declinations, ray_angles, day_angles):
        recorded_year.append(state)
        yield state


show_animation(
    render.year_animation(nodus, foot, seasons, days, hour_suns, hours, envelope_rays, streamed_year()),
    "sundial_year",
    frame_duration_ms,
    n_frames=n_half_year,
    forward_then_back=True,
);

On the left, golden daytime rays pass from the sky through the gnomon's nodus onto the cream dial plate and shorten against the plate's vertical border walls where shadows leave the face; on the right, the orange active day's conic glides over the faint seasonal family between the winter and summer solstice branches.

In [ ]:
# checks
assert len(recorded_year) == n_half_year
# The noon sun stands at a quarter turn minus the latitude plus the declination above the horizon.
expected_heights = np.sin(np.pi / 2 - latitude + solstices_and_equinox)
assert (((dial & noon_suns) - expected_heights).abs() < 1e-10).all()
# Every shadow ray of a season's day lies on its cone, and each hour line follows the gnomonic angle.
assert ((cones[:, None](rays) & rays).abs() < 1e-10).all()
expected_east = np.sin(latitude) * np.sin(hour_angles)[:, None]
expected_north = np.cos(hour_angles)[:, None]
assert ((expected_north * east_offsets - expected_east * north_offsets).abs() < 1e-10).all()